# Synthetic Hourly Sin/Cos Data Generator

Генератор минутных данных за период с `2020-02-01` до `2026-02-01` включительно.
Для каждой минуты считается `cos(2πt/T)` и `sin(2πt/T)`, где:
- `t = hour + minute / 60`
- `T = 24`
- `weekday = timestamp.weekday()` (0 = понедельник, 6 = воскресенье)
- `t_week = weekday + hour / 24`
- `cos_weekday = cos(2π * t_week / 7)`
- `sin_weekday = sin(2π * t_week / 7)`

Данные сохраняются в отдельные parquet-файлы по дням, чтобы их можно было удобно объединить по дате или по `timestamp` с другими дневными наборами, например `klines_ada_restored.parquet`.


In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

In [12]:
import yaml
import boto3

with open("config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

print('bucket =', cfg['storage']['bucket'])
print('prefix =', cfg['storage']['prefix'])
print('symbols =', cfg['symbols'])
print('S3 env:', bool(os.getenv('YC_ENDPOINT')), bool(os.getenv('YC_REGION')), bool(os.getenv('YC_ACCESS_KEY_ID')), bool(os.getenv('YC_SECRET_ACCESS_KEY')))

s3 = boto3.client(
    's3',
    endpoint_url=os.getenv('YC_ENDPOINT'),
    region_name=os.getenv('YC_REGION'),
    aws_access_key_id=os.getenv('YC_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('YC_SECRET_ACCESS_KEY'),
)
resp = s3.list_objects_v2(
    Bucket=cfg['storage']['bucket'],
    Prefix=f"{cfg['storage']['prefix']}/klines/symbol={cfg['symbols'][0]}/interval=1m/",
    MaxKeys=1,
)
print('list objects result:', resp.get('KeyCount'))

bucket = binance-data-downloader
prefix = raw
symbols = ['ADAUSDT']
S3 env: True True True True
list objects result: 1


In [ ]:
# Copy current klines files to a new S3 directory so the current data is preserved.
source_prefix = f"{cfg['storage']['prefix']}/klines/symbol={SYMBOL}/interval={INTERVAL}/"
dest_prefix = f"{cfg['storage']['prefix']}/klines_backup/symbol={SYMBOL}/interval={INTERVAL}/"

paginator = s3.get_paginator('list_objects_v2')
page_iterator = paginator.paginate(Bucket=cfg['storage']['bucket'], Prefix=source_prefix)

copied = 0
for page in page_iterator:
    for obj in page.get('Contents', []):
        source_key = obj['Key']
        dest_key = source_key.replace(source_prefix, dest_prefix, 1)
        s3.copy_object(
            Bucket=cfg['storage']['bucket'],
            CopySource={'Bucket': cfg['storage']['bucket'], 'Key': source_key},
            Key=dest_key,
        )
        copied += 1
        if copied % 100 == 0:
            print(f'Copied {copied} objects so far...')

print(f'Copy complete, total objects copied: {copied}')

In [3]:
# Параметры генерации
START_DATE = "2020-02-01"
END_DATE = "2026-02-01"
OUTPUT_DIR = Path("synthetic_hourly")
T = 24

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("output dir:", OUTPUT_DIR.resolve())

output dir: C:\projects\binance-dowloader-3.0\synthetic_hourly


In [8]:
def build_daily_synthetic_day(date: str, output_dir: Path) -> Path:
    """Собирает минутные данные для одного дня и сохраняет их в parquet."""
    date_start = pd.Timestamp(date).tz_localize("UTC")
    date_end = date_start + pd.Timedelta(days=1) - pd.Timedelta(minutes=1)

    timestamps = pd.date_range(
        start=date_start,
        end=date_end,
        freq="min",
        tz="UTC",
        inclusive="both",
    )

    df = pd.DataFrame({
        "timestamp": timestamps,
        "hour": timestamps.hour,
        "minute": timestamps.minute,
        "weekday": timestamps.weekday,
    })

    df["t"] = df["hour"] + df["minute"] / 60
    df["cos_hour"] = np.cos(2 * np.pi * df["t"] / T)
    df["sin_hour"] = np.sin(2 * np.pi * df["t"] / T)
    df["t_week"] = df["weekday"] + df["hour"] / 24
    df["cos_weekday"] = np.cos(2 * np.pi * df["t_week"] / 7)
    df["sin_weekday"] = np.sin(2 * np.pi * df["t_week"] / 7)
    df["is_weekend"] = (df["weekday"] >= 5).astype(int)

    df = add_exchange_open_flags(df)

    output_path = output_dir / f"synthetic_hourly_{date}.parquet"
    df.to_parquet(output_path, index=False)
    return output_path


# Пример: создаем файл только для одного дня

example_date = "2020-02-03"

example_path = build_daily_synthetic_day(example_date, OUTPUT_DIR)
print("Saved example file:", example_path)

Saved example file: synthetic_hourly\synthetic_hourly_2020-02-03.parquet


In [4]:
def add_exchange_open_flags(df: pd.DataFrame) -> pd.DataFrame:
    """Добавляет флаги открытия/закрытия для групп бирж: Asia, Europe, US."""
    trading_day = df["weekday"] < 5

    df["asia_open"] = (
        trading_day
        & df["hour"].between(0, 8)
    ).astype(int)
    df["europe_open"] = (
        trading_day
        & df["hour"].between(7, 16)
    ).astype(int)
    df["us_open"] = (
        trading_day
        & (
            ((df["hour"] == 13) & (df["minute"] >= 30))
            | df["hour"].between(14, 19)
            | ((df["hour"] == 20) & (df["minute"] == 0))
        )
    ).astype(int)

    return df

In [ ]:
import yaml

from s3_writer import S3Writer

with open("config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

s3_writer = S3Writer(bucket=cfg["storage"]["bucket"], prefix=cfg["storage"]["prefix"])

SYMBOL = cfg["symbols"][0]
SOURCE = "sinthetic_data"
INTERVAL = "1m"


def build_daily_synthetic_df(date: str) -> pd.DataFrame:
    date_start = pd.Timestamp(date).tz_localize("UTC")
    date_end = date_start + pd.Timedelta(days=1) - pd.Timedelta(minutes=1)

    timestamps = pd.date_range(
        start=date_start,
        end=date_end,
        freq="min",
        tz="UTC",
        inclusive="both",
    )

    df = pd.DataFrame({
        "timestamp": timestamps,
        "hour": timestamps.hour,
        "minute": timestamps.minute,
        "weekday": timestamps.weekday,
    })

    df["t"] = df["hour"] + df["minute"] / 60
    df["cos_hour"] = np.cos(2 * np.pi * df["t"] / T)
    df["sin_hour"] = np.sin(2 * np.pi * df["t"] / T)
    df["t_week"] = df["weekday"] + df["hour"] / 24
    df["cos_weekday"] = np.cos(2 * np.pi * df["t_week"] / 7)
    df["sin_weekday"] = np.sin(2 * np.pi * df["t_week"] / 7)
    df["is_weekend"] = (df["weekday"] >= 5).astype(int)
    df = add_exchange_open_flags(df)

    return df


# Upload synthetic minute data for the full configured range to S3.
# This will write files as raw/klines/symbol=<SYMBOL>/interval=1m/date=<YYYY-MM-DD>/data.parquet

full_dates = pd.date_range(start=cfg["date_range"]["start"], end=cfg["date_range"]["end"], freq="D")
print(f"Uploading synthetic data for {len(full_dates)} days to s3://{cfg['storage']['bucket']}/{cfg['storage']['prefix']}/...")

for date in full_dates.strftime("%Y-%m-%d"):
    df = build_daily_synthetic_df(date)
    s3_writer.write_df(df, source=SOURCE, symbol=SYMBOL, interval=INTERVAL, date=date)

print("Finished uploading all synthetic minute files to S3.")

Uploading synthetic data for 2193 days to s3://binance-data-downloader/raw/...
Uploaded: s3://binance-data-downloader/raw/klines/symbol=ADAUSDT/interval=1m/date=2020-02-01/data.parquet
Uploaded: s3://binance-data-downloader/raw/klines/symbol=ADAUSDT/interval=1m/date=2020-02-02/data.parquet
Uploaded: s3://binance-data-downloader/raw/klines/symbol=ADAUSDT/interval=1m/date=2020-02-03/data.parquet
Uploaded: s3://binance-data-downloader/raw/klines/symbol=ADAUSDT/interval=1m/date=2020-02-04/data.parquet
Uploaded: s3://binance-data-downloader/raw/klines/symbol=ADAUSDT/interval=1m/date=2020-02-05/data.parquet
Uploaded: s3://binance-data-downloader/raw/klines/symbol=ADAUSDT/interval=1m/date=2020-02-06/data.parquet
Uploaded: s3://binance-data-downloader/raw/klines/symbol=ADAUSDT/interval=1m/date=2020-02-07/data.parquet
Uploaded: s3://binance-data-downloader/raw/klines/symbol=ADAUSDT/interval=1m/date=2020-02-08/data.parquet
Uploaded: s3://binance-data-downloader/raw/klines/symbol=ADAUSDT/interval

In [9]:
# Проверяем содержимое созданного файла
example_df = pd.read_parquet(example_path)
print(example_df.head(8))
print(example_df.tail(3))
print("rows:", len(example_df))
print("timestamp range:", example_df["timestamp"].min(), "-", example_df["timestamp"].max())
print("unique hours:", sorted(example_df["hour"].unique()))

                  timestamp  hour  minute  weekday         t  cos_hour  \
0 2020-02-03 00:00:00+00:00     0       0        0  0.000000  1.000000   
1 2020-02-03 00:01:00+00:00     0       1        0  0.016667  0.999990   
2 2020-02-03 00:02:00+00:00     0       2        0  0.033333  0.999962   
3 2020-02-03 00:03:00+00:00     0       3        0  0.050000  0.999914   
4 2020-02-03 00:04:00+00:00     0       4        0  0.066667  0.999848   
5 2020-02-03 00:05:00+00:00     0       5        0  0.083333  0.999762   
6 2020-02-03 00:06:00+00:00     0       6        0  0.100000  0.999657   
7 2020-02-03 00:07:00+00:00     0       7        0  0.116667  0.999534   

   sin_hour  t_week  cos_weekday  sin_weekday  is_weekend  asia_open  \
0  0.000000     0.0          1.0          0.0           0          1   
1  0.004363     0.0          1.0          0.0           0          1   
2  0.008727     0.0          1.0          0.0           0          1   
3  0.013090     0.0          1.0          0.0

In [10]:
example_df.describe()

,hour,minute,weekday,t,cos_hour,sin_hour,t_week,cos_weekday,sin_weekday,is_weekend,asia_open,europe_open,us_open
count,1440.000000,1440.000000,1440.0,1440.000000,1.440000e+03,1.440000e+03,1440.000000,1440.000000,1440.000000,1440.0,1440.000000,1440.000000,1440.000000
mean,11.500000,29.500000,0.0,11.991667,-1.973730e-17,0.000000e+00,0.479167,0.878769,0.403127,0.0,0.375000,0.416667,0.271528
std,6.924591,17.324119,0.0,6.930608,7.073524e-01,7.073524e-01,0.288525,0.109177,0.231040,0.0,0.484291,0.493178,0.444902
min,0.000000,0.000000,0.0,0.000000,-1.000000e+00,-1.000000e+00,0.000000,0.652287,0.000000,0.0,0.000000,0.000000,0.000000
25%,5.750000,14.750000,0.0,5.995833,-7.071068e-01,-7.071068e-01,0.239583,0.798906,0.213369,0.0,0.000000,0.000000,0.000000
50%,11.500000,29.500000,0.0,11.991667,-6.123234e-17,6.123234e-17,0.479167,0.908766,0.416888,0.0,0.000000,0.000000,0.000000
75%,17.250000,44.250000,0.0,17.987500,7.071068e-01,7.071068e-01,0.718750,0.976838,0.601238,0.0,1.000000,1.000000,1.000000
max,23.000000,59.000000,0.0,23.983333,1.000000e+00,1.000000e+00,0.958333,1.000000,0.757972,0.0,1.000000,1.000000,1.000000


In [11]:
example_df

,timestamp,hour,minute,weekday,t,cos_hour,sin_hour,t_week,cos_weekday,sin_weekday,is_weekend,asia_open,europe_open,us_open
0,2020-02-03 00:00:00+00:00,0,0,0,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0,1,0,0
1,2020-02-03 00:01:00+00:00,0,1,0,0.016667,0.999990,0.004363,0.000000,1.000000,0.000000,0,1,0,0
2,2020-02-03 00:02:00+00:00,0,2,0,0.033333,0.999962,0.008727,0.000000,1.000000,0.000000,0,1,0,0
3,2020-02-03 00:03:00+00:00,0,3,0,0.050000,0.999914,0.013090,0.000000,1.000000,0.000000,0,1,0,0
4,2020-02-03 00:04:00+00:00,0,4,0,0.066667,0.999848,0.017452,0.000000,1.000000,0.000000,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1435,2020-02-03 23:55:00+00:00,23,55,0,23.916667,0.999762,-0.021815,0.958333,0.652287,0.757972,0,0,0,0
1436,2020-02-03 23:56:00+00:00,23,56,0,23.933333,0.999848,-0.017452,0.958333,0.652287,0.757972,0,0,0,0
1437,2020-02-03 23:57:00+00:00,23,57,0,23.950000,0.999914,-0.013090,0.958333,0.652287,0.757972,0,0,0,0
1438,2020-02-03 23:58:00+00:00,23,58,0,23.966667,0.999962,-0.008727,0.958333,0.652287,0.757972,0,0,0,0


In [11]:
# Генерация файлов для первых 7 дней
# Оставляем только первые 7 дат для быстрой проверки.

all_dates = pd.date_range(start=pd.Timestamp(START_DATE), periods=7, freq="D")

for date in all_dates.strftime("%Y-%m-%d"):
    path = OUTPUT_DIR / f"synthetic_hourly_{date}.parquet"
    if not path.exists():
        build_daily_synthetic_day(date, OUTPUT_DIR)
    print("Saved:", path)

print("Completed generation for", len(all_dates), "days")

Saved: synthetic_hourly\synthetic_hourly_2020-02-01.parquet
Saved: synthetic_hourly\synthetic_hourly_2020-02-02.parquet
Saved: synthetic_hourly\synthetic_hourly_2020-02-03.parquet
Saved: synthetic_hourly\synthetic_hourly_2020-02-04.parquet
Saved: synthetic_hourly\synthetic_hourly_2020-02-05.parquet
Saved: synthetic_hourly\synthetic_hourly_2020-02-06.parquet
Saved: synthetic_hourly\synthetic_hourly_2020-02-07.parquet
Completed generation for 7 days


In [12]:
# Объединяем все 7 файлов в один DataFrame
merged_frames = []
for date in all_dates.strftime("%Y-%m-%d"):
    file_path = OUTPUT_DIR / f"synthetic_hourly_{date}.parquet"
    merged_frames.append(pd.read_parquet(file_path))

merged_df = pd.concat(merged_frames, ignore_index=True)
print("Merged rows:", len(merged_df))
print("Unique dates:", merged_df["timestamp"].dt.date.nunique())
print("Timestamp range:", merged_df["timestamp"].min(), "-", merged_df["timestamp"].max())

merged_output = OUTPUT_DIR / "synthetic_hourly_first_7_days.parquet"
merged_df.to_parquet(merged_output, index=False)
print("Saved merged file:", merged_output)
merged_df.head()

Merged rows: 10080
Unique dates: 7
Timestamp range: 2020-02-01 00:00:00+00:00 - 2020-02-07 23:59:00+00:00
Saved merged file: synthetic_hourly\synthetic_hourly_first_7_days.parquet


,timestamp,hour,minute,weekday,t,cos_hour,sin_hour,cos_weekday,sin_weekday
0,2020-02-01 00:00:00+00:00,0,0,5,0.000000,1.000000,0.000000,-0.222521,-0.974928
1,2020-02-01 00:01:00+00:00,0,1,5,0.016667,0.999990,0.004363,-0.222521,-0.974928
2,2020-02-01 00:02:00+00:00,0,2,5,0.033333,0.999962,0.008727,-0.222521,-0.974928
3,2020-02-01 00:03:00+00:00,0,3,5,0.050000,0.999914,0.013090,-0.222521,-0.974928
4,2020-02-01 00:04:00+00:00,0,4,5,0.066667,0.999848,0.017452,-0.222521,-0.974928


## Как объединять с другими данными

Если нужно объединить с набором `klines_ada_restored.parquet`, можно сделать это по столбцу `timestamp`.

Пример:

```python
base_df = pd.read_parquet("klines_ada_restored.parquet")
synthetic_df = pd.read_parquet("synthetic_hourly/synthetic_hourly_2020-02-01.parquet")
merged = base_df.merge(synthetic_df, on="timestamp", how="left")
```

Если часы уже присутствуют в `klines_ada_restored.parquet`, синус/косинус будут добавлены к каждой минуте.